# Sistema Especialista Fuzzy de Triagem de Resíduos

Este notebook é a versão autocontida do projeto original por níveis. Ele usa um controlador **Mamdani**, implementado com `scikit-fuzzy`, para decidir a destinação de materiais recicláveis a partir da contaminação e da umidade.

Execute as células na ordem. A célula de instalação pode ser ignorada se as dependências já estiverem disponíveis.

In [ ]:
# Execute uma vez no Jupyter/Colab.
%pip install -q numpy scipy networkx scikit-fuzzy matplotlib

## Domínio, entradas e saída

- **Contaminação (0–100%)**: baixa, média e alta.
- **Umidade (0–100%)**: seca, moderada e alta.
- **Destinação (0–100)**: rejeito, triagem e reciclagem.

Os conjuntos se sobrepõem para que valores próximos às fronteiras não provoquem mudanças abruptas. Resíduos perigosos e orgânicos são exceções de domínio: logística reversa e compostagem, respectivamente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl

CORES_RECICLAVEIS = {
    'plastico': 'VERMELHA',
    'papel': 'AZUL',
    'vidro': 'VERDE',
    'metal': 'AMARELA',
}
MATERIAIS_PERIGOSOS = {'pilha', 'bateria', 'eletronico', 'lampada', 'remedio'}

universo = np.arange(0, 101, 1, dtype=float)
contaminacao = ctrl.Antecedent(universo, 'contaminacao')
umidade = ctrl.Antecedent(universo, 'umidade')
destinacao = ctrl.Consequent(universo, 'destinacao')

# Funções de pertinência: três termos por variável.
contaminacao['baixa'] = fuzz.trapmf(universo, [0, 0, 15, 35])
contaminacao['media'] = fuzz.trimf(universo, [20, 50, 80])
contaminacao['alta'] = fuzz.trapmf(universo, [65, 85, 100, 100])

umidade['seca'] = fuzz.trapmf(universo, [0, 0, 20, 40])
umidade['moderada'] = fuzz.trimf(universo, [25, 50, 75])
umidade['alta'] = fuzz.trapmf(universo, [60, 80, 100, 100])

destinacao['rejeito'] = fuzz.trapmf(universo, [0, 0, 20, 45])
destinacao['triagem'] = fuzz.trimf(universo, [30, 50, 70])
destinacao['reciclagem'] = fuzz.trapmf(universo, [55, 75, 100, 100])

## Base de regras fuzzy

As nove regras abaixo representam todas as combinações da matriz 3 × 3 das entradas; portanto, o espaço de entrada fica coberto sem lacunas.

In [ ]:
DEFINICOES_REGRAS = [
    ('R01', 'baixa', 'seca', 'reciclagem'),
    ('R02', 'baixa', 'moderada', 'triagem'),
    ('R03', 'baixa', 'alta', 'rejeito'),
    ('R04', 'media', 'seca', 'reciclagem'),
    ('R05', 'media', 'moderada', 'triagem'),
    ('R06', 'media', 'alta', 'rejeito'),
    ('R07', 'alta', 'seca', 'rejeito'),
    ('R08', 'alta', 'moderada', 'rejeito'),
    ('R09', 'alta', 'alta', 'rejeito'),
]

for identificador, termo_contaminacao, termo_umidade, termo_destinacao in DEFINICOES_REGRAS:
    print(f'{identificador}: SE contaminação é {termo_contaminacao} E umidade é {termo_umidade}, ENTÃO destinação é {termo_destinacao}.')

regras = [
    ctrl.Rule(
        contaminacao[termo_contaminacao] & umidade[termo_umidade],
        destinacao[termo_destinacao],
        label=identificador,
    )
    for identificador, termo_contaminacao, termo_umidade, termo_destinacao in DEFINICOES_REGRAS
]
sistema_mamdani = ctrl.ControlSystem(regras)

In [ ]:
def validar_percentual(nome, valor):
    valor = float(valor)
    if not 0 <= valor <= 100:
        raise ValueError(f'{nome} deve estar entre 0 e 100.')
    return valor

def graus(variavel, valor):
    return {
        termo: float(fuzz.interp_membership(variavel.universe, variavel[termo].mf, valor))
        for termo in variavel.terms
    }

def avaliar_fuzzy(valor_contaminacao, valor_umidade):
    valor_contaminacao = validar_percentual('Contaminação', valor_contaminacao)
    valor_umidade = validar_percentual('Umidade', valor_umidade)

    simulacao = ctrl.ControlSystemSimulation(sistema_mamdani, cache=False)
    simulacao.input['contaminacao'] = valor_contaminacao
    simulacao.input['umidade'] = valor_umidade
    simulacao.compute()
    escore = float(simulacao.output['destinacao'])

    graus_contaminacao = graus(contaminacao, valor_contaminacao)
    graus_umidade = graus(umidade, valor_umidade)
    regras_ativas = []
    for identificador, termo_contaminacao, termo_umidade, _ in DEFINICOES_REGRAS:
        forca = min(graus_contaminacao[termo_contaminacao], graus_umidade[termo_umidade])
        if forca > 0:
            regras_ativas.append((identificador, round(forca, 3)))

    saida = {
        termo: round(float(fuzz.interp_membership(destinacao.universe, destinacao[termo].mf, escore)), 3)
        for termo in destinacao.terms
    }
    return escore, saida, sorted(regras_ativas, key=lambda item: item[1], reverse=True)

def triar(material, valor_contaminacao, valor_umidade):
    material = material.strip().lower()
    valor_contaminacao = validar_percentual('Contaminação', valor_contaminacao)
    valor_umidade = validar_percentual('Umidade', valor_umidade)

    if material in MATERIAIS_PERIGOSOS:
        return {
            'classe': 'perigoso', 'lixeira': 'LARANJA', 'rota': 'logística reversa',
            'escore': None, 'regras_ativas': [],
            'mensagem': 'Regra de segurança: material perigoso vai a ponto de coleta especial.',
        }
    if material == 'organico':
        return {
            'classe': 'compostavel', 'lixeira': 'MARROM', 'rota': 'compostagem',
            'escore': None, 'regras_ativas': [],
            'mensagem': 'Regra de domínio: material orgânico segue para compostagem.',
        }
    if material not in CORES_RECICLAVEIS:
        raise ValueError(f'Material desconhecido: {material}.')

    escore, pertinencias_saida, regras_ativas = avaliar_fuzzy(valor_contaminacao, valor_umidade)
    if escore >= 70:
        classe, lixeira, rota = 'reciclavel', CORES_RECICLAVEIS[material], 'coleta seletiva / reciclagem'
        mensagem = 'Qualidade predominantemente reciclável.'
    elif escore >= 40:
        classe, lixeira, rota = 'triagem', 'TRIAGEM', 'inspeção, limpeza e nova avaliação'
        mensagem = 'Há incerteza: encaminhar para inspeção e limpeza antes da decisão final.'
    else:
        classe, lixeira, rota = 'rejeito', 'CINZA', 'aterro sanitário'
        mensagem = 'Qualidade predominantemente de rejeito.'

    return {
        'classe': classe, 'lixeira': lixeira, 'rota': rota, 'escore': round(escore, 2),
        'pertinencias_saida': pertinencias_saida, 'regras_ativas': regras_ativas, 'mensagem': mensagem,
    }

def mostrar_resultado(titulo, material, valor_contaminacao, valor_umidade):
    resultado = triar(material, valor_contaminacao, valor_umidade)
    print()
    print(f'CASO: {titulo}')
    print(f'Entrada: material={material} | contaminação={valor_contaminacao}% | umidade={valor_umidade}%')
    if resultado['escore'] is not None:
        print('Saída Mamdani (centroide): {:.2f}/100'.format(resultado['escore']))
        print('Pertinência da saída: {}'.format(resultado['pertinencias_saida']))
        print('Regras fuzzy ativas: {}'.format(resultado['regras_ativas']))
    else:
        print('Saída Mamdani: não aplicada (exceção de segurança/domínio).')
    print('Decisão: {} | lixeira {} | rota: {}'.format(resultado['classe'].upper(), resultado['lixeira'], resultado['rota']))
    print('Justificativa: {}'.format(resultado['mensagem']))
    return resultado

In [ ]:
# Demonstração dos mesmos três casos do projeto original.
mostrar_resultado('Garrafa PET limpa e seca', 'plastico', 5, 10)
mostrar_resultado('Papel limpo, porém molhado', 'papel', 10, 70)
mostrar_resultado('Pilha usada (resíduo perigoso)', 'pilha', 0, 0)

In [ ]:
# Visualização das funções de pertinência para apresentação e relatório.
figura, eixos = plt.subplots(3, 1, figsize=(10, 10), constrained_layout=True)
for eixo, variavel, titulo in (
    (eixos[0], contaminacao, 'Contaminação'),
    (eixos[1], umidade, 'Umidade'),
    (eixos[2], destinacao, 'Destinação'),
):
    for termo, conjunto in variavel.terms.items():
        eixo.plot(variavel.universe, conjunto.mf, linewidth=2, label=termo.capitalize())
    eixo.set_title(titulo)
    eixo.set_xlim(0, 100)
    eixo.set_ylim(-0.05, 1.05)
    eixo.set_xlabel('Percentual / escore')
    eixo.set_ylabel('Grau de pertinência')
    eixo.grid(alpha=0.25)
    eixo.legend(loc='upper right')
plt.show()

## Comparação com o projeto original

No projeto original, uma regra por nível vence pelo uso de limites rígidos, `salience` e `NOT(Estado())`. Nesta versão, mais de uma regra pode ser ativada ao mesmo tempo e o centroide compõe a saída. Isso aumenta a complexidade de projeto — é necessário justificar curvas, sobreposições e regras —, mas melhora o poder de expressar incerteza e casos limítrofes. Regras clássicas foram mantidas onde a prioridade é absoluta: resíduos perigosos e compostagem.